# C04d — Step 10.9: Genre-conditioned ORG repair

Exploratory follow-up after accepted Step 10 CARDINAL.

The Step-10 audit shows that residual `ner_ORG_*` direction is **genre-specific**, not globally monotone. This notebook therefore uses fixed same-genre human references and routes different genres to different ORG operations.

Operations:
- **ADD_COREF** — cross-sentence coreference explicitization.
- **ADD_RESTORE** — source-grounded ORG restoration.
- **FUSE** — fuse adjacent ORG-bearing sentences while preserving the entity multiset.
- **REMOVE** — de-explicitize a repeated ORG mention into a reduced reference.
- **SPLIT** — split an ORG-dense sentence while preserving the entity multiset.

Candidate selection uses only fixed human Q20--Q80 ORG bands, preservation guards, and a pinned likelihood scorer. Frozen FZ301 is evaluated only after a candidate endpoint is accepted.


In [1]:
from pathlib import Path
import json, re, hashlib, pickle
from collections import Counter
from functools import lru_cache
import numpy as np, pandas as pd
import spacy, torch
import torch.nn.functional as F
from IPython.display import display
from transformers import AutoTokenizer, AutoModelForCausalLM
from huggingface_hub import model_info

ROOT=Path.cwd() if (Path.cwd()/"results").exists() else Path.cwd().parent
BASE=ROOT/"results/causal/C02d2_fz118_ghostbuster_raid"
C04_ROOT=ROOT/"results/causal/C04_fz301_raid_fixed54"
STEP10_ROOT=ROOT/"results/causal/C04c_CARDINAL_controlled_insertion"
OUT_ROOT=ROOT/"results/causal/C04d_ORG_discourse_step10p9"
SEED=20260912
np.random.seed(SEED); torch.manual_seed(SEED)
nlp=spacy.load("en_core_web_sm")

# Accepted frozen FZ301 baseline.
c04_runs=[]
for p in sorted(C04_ROOT.glob("*")):
    if not (p/"FZ301.pkl").exists() or not (p/"stage_summary.csv").exists(): continue
    s=pd.read_csv(p/"stage_summary.csv")
    if len(s)==3 and np.isclose(float(s.iloc[0].AUROC),.875380,atol=5e-6) and np.isclose(float(s.iloc[2].AUROC),.699558,atol=5e-6): c04_runs.append(p)
assert len(c04_runs)==1, c04_runs
C04_RUN=c04_runs[0]
with (C04_RUN/"FZ301.pkl").open("rb") as f: fz301=pickle.load(f)
C04_MANIFEST=json.loads((C04_RUN/"feature_manifest.json").read_text())

# Accepted Step 10 endpoint.
step10_runs=[]
for p in sorted(STEP10_ROOT.glob("*")):
    if not (p/"stage_summary.csv").exists() or not (p/"texts_after_CARDINAL_step10.csv").exists(): continue
    s=pd.read_csv(p/"stage_summary.csv"); last=s.iloc[-1]
    if np.isclose(float(last.AUROC),.671326,atol=5e-6) and np.isclose(float(last.mean_AI_score),.476821,atol=5e-6): step10_runs.append(p)
assert len(step10_runs)==1, step10_runs
STEP10_RUN=step10_runs[0]
data=pd.read_csv(STEP10_RUN/"texts_after_CARDINAL_step10.csv",keep_default_na=False).sort_values("case_id").reset_index(drop=True)
assert len(data)==54 and data.case_id.is_unique

features=pd.read_parquet(BASE/"fz118_paired_features.parquet")
corpus=pd.read_parquet(BASE/"paired_corpus.parquet")
raid=features.loc[features.dataset.eq("raid")].reset_index(drop=True)
text_map=corpus.set_index("source_id").text.to_dict()

def norm_surface(s): return re.sub(r"\s+"," ",str(s).strip().lower())
def org_stats(text):
    doc=nlp(str(text)); sents=list(doc.sents)
    counts=np.array([sum(e.label_=="ORG" for e in s.ents) for s in sents],float)
    ents=list(doc.ents); n_org=sum(e.label_=="ORG" for e in ents)
    return {"n_ORG":n_org,"n_entities":len(ents),"n_sentences":len(sents),
            "ner_ORG_entity_share":n_org/len(ents) if ents else 0.,
            "ner_ORG_sentence_mean":counts.mean() if len(counts) else 0.,
            "ner_ORG_sentence_std":counts.std(ddof=0) if len(counts) else 0.,
            "ner_ORG_sentence_max":counts.max() if len(counts) else 0.}
ORG_FEATURES=["ner_ORG_entity_share","ner_ORG_sentence_mean","ner_ORG_sentence_std","ner_ORG_sentence_max"]

h=raid.loc[raid.split.eq("test") & raid.label.eq(0) & raid.domain.isin(data.domain.unique()),["source_id","domain"]].copy()
h["text"]=h.source_id.map(text_map)
print("Accepted Step 10:",STEP10_RUN.relative_to(ROOT))


/home/user/Downloads/TemporalContextResearch/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Accepted Step 10: results/causal/C04c_CARDINAL_controlled_insertion/16d25f12a76b


In [2]:
# Genre-conditioned human bands and automatic intervention policy.

def stats_frame(df):
    s=pd.DataFrame([org_stats(t) for t in df.text]); s["domain"]=df.domain.to_numpy()
    return s

H=stats_frame(h); A=stats_frame(data[["domain","text"]])
hmed=H.groupby("domain")[ORG_FEATURES].median()
h20=H.groupby("domain")[ORG_FEATURES].quantile(.20)
h80=H.groupby("domain")[ORG_FEATURES].quantile(.80)
amed=A.groupby("domain")[ORG_FEATURES].median()

rows=[]
for g in sorted(hmed.index):
    for f in ORG_FEATURES:
        rows.append({"domain":g,"feature":f,"AI":amed.loc[g,f],"Human":hmed.loc[g,f],
                     "Human_minus_AI":hmed.loc[g,f]-amed.loc[g,f],
                     "Human_q20":h20.loc[g,f],"Human_q80":h80.loc[g,f]})
genre_medians=pd.DataFrame(rows)
display(genre_medians)

tol={f:.01 for f in ORG_FEATURES}; tol["ner_ORG_sentence_max"]=.5
GENRE_POLICY={}; rows=[]
for g in sorted(amed.index):
    dirs={}
    for f in ORG_FEATURES:
        d=float(hmed.loc[g,f]-amed.loc[g,f])
        dirs[f]="UP" if d>tol[f] else "DOWN" if d<-tol[f] else "HOLD"
    ops=[]
    if any(dirs[f]=="UP" for f in ["ner_ORG_entity_share","ner_ORG_sentence_mean"]): ops.append("ADD")
    if any(dirs[f]=="DOWN" for f in ["ner_ORG_entity_share","ner_ORG_sentence_mean"]): ops.append("REMOVE")
    if any(dirs[f]=="UP" for f in ["ner_ORG_sentence_std","ner_ORG_sentence_max"]): ops.append("FUSE")
    if any(dirs[f]=="DOWN" for f in ["ner_ORG_sentence_std","ner_ORG_sentence_max"]): ops.append("SPLIT")
    GENRE_POLICY[g]={"dirs":dirs,"ops":ops}
    rows.append({"domain":g,**{f.replace("ner_ORG_",""):dirs[f] for f in ORG_FEATURES},
                 "operations":" + ".join(ops) if ops else "HOLD"})
policy_df=pd.DataFrame(rows)
display(policy_df)

def inside_band(x,domain,f):
    return float(h20.loc[domain,f])-1e-12 <= float(x) <= float(h80.loc[domain,f])+1e-12

def org_band_distance(text,domain):
    st=org_stats(text); total=0.
    for f in ORG_FEATURES:
        lo,hi=float(h20.loc[domain,f]),float(h80.loc[domain,f]); x=float(st[f])
        scale=max(hi-lo,1.0 if f=="ner_ORG_sentence_max" else max(abs(float(hmed.loc[domain,f]))*.25,.05))
        total+=max(lo-x,0,x-hi)/scale
    return total


,domain,feature,AI,Human,Human_minus_AI,Human_q20,Human_q80
0,abstracts,ner_ORG_entity_share,0.361111,0.400000,0.038889,0.000000,0.711364
1,abstracts,ner_ORG_sentence_mean,0.366667,0.375000,0.008333,0.000000,0.695238
2,abstracts,ner_ORG_sentence_std,0.480651,0.489898,0.009247,0.000000,0.933781
3,abstracts,ner_ORG_sentence_max,1.000000,1.000000,0.000000,0.000000,3.000000
4,books,ner_ORG_entity_share,0.000000,0.175192,0.175192,0.020000,0.306355
5,books,ner_ORG_sentence_mean,0.000000,0.436508,0.436508,0.040000,0.571429
6,books,ner_ORG_sentence_std,0.000000,0.476565,0.476565,0.120000,0.705687
7,books,ner_ORG_sentence_max,0.000000,1.000000,1.000000,0.400000,2.000000
8,news,ner_ORG_entity_share,0.150997,0.176471,0.025473,0.088727,0.275000
9,news,ner_ORG_sentence_mean,0.266667,0.285714,0.019048,0.161905,0.617460


,domain,entity_share,sentence_mean,sentence_std,sentence_max,operations
0,abstracts,UP,HOLD,HOLD,HOLD,ADD
1,books,UP,UP,UP,UP,ADD + FUSE
2,news,UP,UP,UP,HOLD,ADD + FUSE
3,recipes,DOWN,DOWN,DOWN,HOLD,REMOVE + SPLIT
4,reddit,HOLD,HOLD,HOLD,HOLD,HOLD
5,reviews,UP,UP,UP,UP,ADD + FUSE
6,wiki,DOWN,HOLD,DOWN,DOWN,REMOVE + SPLIT


In [3]:
# Coverage manifests for ADD/FUSE plus alias audit.

REF_TERMS={"it","its","they","their","them","company","firm","organization","organisation","agency","corporation",
           "group","association","committee","university","institute","institution","bank","team","publisher","network"}

def sent_orgs(sent): return [e.text for e in sent.ents if e.label_=="ORG"]
def content_lemmas(text):
    d=nlp(str(text))
    return {t.lemma_.lower() for t in d if t.is_alpha and not t.is_stop and t.pos_ in {"NOUN","PROPN","VERB","ADJ","ADV"}}

fusion_rows=[]; coref_rows=[]; restore_rows=[]; alias_rows=[]
for r in data.itertuples(index=False):
    doc=nlp(r.text); sents=list(doc.sents)
    for i in range(len(sents)-1):
        a,b=sents[i],sents[i+1]; oa,ob=sent_orgs(a),sent_orgs(b)
        if oa and ob and len((a.text+" "+b.text).split())<=60:
            fusion_rows.append({"case_id":r.case_id,"domain":r.domain,"sent_i":i,"start":a.start_char,"end":b.end_char,
                                "sentence_a":a.text,"sentence_b":b.text,"orgs":" | ".join(oa+ob)})
        refs=sorted({t.text for t in b if t.lemma_.lower() in REF_TERMS})
        if len(set(map(str.lower,oa)))==1 and not ob and refs:
            coref_rows.append({"case_id":r.case_id,"domain":r.domain,"sent_i":i+1,"start":b.start_char,"end":b.end_char,
                               "context_sentence":a.text,"source_sentence":b.text,"target_org":oa[-1],"reference_terms":" | ".join(refs)})
    src_doc=nlp(text_map[r.source_id]); current_orgs={e.text.lower() for e in doc.ents if e.label_=="ORG"}
    for ss in src_doc.sents:
        for ent in [e for e in ss.ents if e.label_=="ORG" and e.text.lower() not in current_orgs]:
            A0=content_lemmas(ss.text)-content_lemmas(ent.text); best=None
            for j,cs in enumerate(doc.sents):
                if ent.text.lower() in cs.text.lower(): continue
                B0=content_lemmas(cs.text)
                if len(A0&B0)<4: continue
                jac=len(A0&B0)/max(1,len(A0|B0))
                if best is None or jac>best[0]: best=(jac,j,cs)
            if best and best[0]>=.45:
                restore_rows.append({"case_id":r.case_id,"domain":r.domain,"sent_i":best[1],"start":best[2].start_char,"end":best[2].end_char,
                                     "target_org":ent.text,"original_sentence":ss.text,"source_sentence":best[2].text,"jaccard":best[0]})
    for ent in [e for e in doc.ents if e.label_=="ORG"]:
        words=[w for w in re.findall(r"[A-Za-z]+",ent.text) if w.lower() not in {"the","of","and","for","in"}]
        if len(words)>=2:
            ac="".join(w[0].upper() for w in words)
            if 2<=len(ac)<=8:
                alias_rows.append({"case_id":r.case_id,"domain":r.domain,"org":ent.text,"derived_acronym":ac,
                                   "acronym_already_present":bool(re.search(rf"\b{re.escape(ac)}\b",r.text))})

fusion_manifest=pd.DataFrame(fusion_rows); coref_manifest=pd.DataFrame(coref_rows)
restore_manifest=pd.DataFrame(restore_rows); alias_audit=pd.DataFrame(alias_rows)
print("fusion:",len(fusion_manifest),"pairs /",fusion_manifest.case_id.nunique() if len(fusion_manifest) else 0,"cases")
print("coref:",len(coref_manifest),"pairs /",coref_manifest.case_id.nunique() if len(coref_manifest) else 0,"cases")
print("restore:",len(restore_manifest),"alignments /",restore_manifest.case_id.nunique() if len(restore_manifest) else 0,"cases")
print("alias:",len(alias_audit),"ORGs /",alias_audit.case_id.nunique() if len(alias_audit) else 0,"cases")


fusion: 28 pairs / 17 cases
coref: 13 pairs / 12 cases
restore: 10 alignments / 9 cases
alias: 84 ORGs / 31 cases


In [4]:
# Shared pinned Qwen editor + likelihood scorer.

QWEN_ID="Qwen/Qwen2.5-1.5B-Instruct"; QWEN_REV=model_info(QWEN_ID).sha
device="cuda" if torch.cuda.is_available() else "cpu"
qtok=AutoTokenizer.from_pretrained(QWEN_ID,revision=QWEN_REV,use_fast=True)
if qtok.pad_token_id is None: qtok.pad_token=qtok.eos_token
qwen=AutoModelForCausalLM.from_pretrained(QWEN_ID,revision=QWEN_REV,torch_dtype=torch.float16 if device=="cuda" else torch.float32,
                                          low_cpu_mem_usage=True).to(device).eval()

c04b_contracts=sorted((ROOT/"results/causal/C04b_ORG_referential_explicitization").glob("*/contract.json"))
scorer_id="gpt2"; scorer_rev=None
for p in c04b_contracts:
    c=json.loads(p.read_text())
    if "scorer" in c: scorer_id=c["scorer"]; scorer_rev=c.get("scorer_revision"); break
stok=AutoTokenizer.from_pretrained(scorer_id,revision=scorer_rev,use_fast=True)
slm=AutoModelForCausalLM.from_pretrained(scorer_id,revision=scorer_rev).to(device).eval()
sctx=getattr(slm.config,"n_positions",getattr(slm.config,"max_position_embeddings",1024))

@lru_cache(maxsize=8192)
def mean_logp(text):
    enc=stok(str(text),return_tensors="pt",add_special_tokens=False); ids=enc.input_ids
    if ids.shape[1]<2 or ids.shape[1]>sctx: return -np.inf
    ids=ids.to(device)
    with torch.inference_mode():
        logits=slm(ids).logits[:,:-1,:]
        lp=F.log_softmax(logits,-1).gather(-1,ids[:,1:].unsqueeze(-1)).squeeze(-1)
    return float(lp.mean())

def generate(prompt,n=8,max_new=160):
    msgs=[{"role":"system","content":"You are a precise text editor. Return only the requested rewritten text, or NO_CHANGE."},
          {"role":"user","content":prompt}]
    txt=qtok.apply_chat_template(msgs,tokenize=False,add_generation_prompt=True)
    enc=qtok(txt,return_tensors="pt",add_special_tokens=False).to(device); L=enc.input_ids.shape[1]
    with torch.inference_mode():
        out=qwen.generate(**enc,num_beams=n,num_return_sequences=n,max_new_tokens=max_new,do_sample=False,early_stopping=True,
                          pad_token_id=qtok.pad_token_id,eos_token_id=qtok.eos_token_id)
    return list(dict.fromkeys(x.strip().strip('"') for x in qtok.batch_decode(out[:,L:],skip_special_tokens=True)))

def entity_counter(text): return Counter((e.label_,norm_surface(e.text)) for e in nlp(str(text)).ents)
def digits(text): return Counter(re.findall(r"\b\d[\d,./:%-]*\b",str(text)))
def verbs(text): return Counter(t.lemma_.lower() for t in nlp(str(text)) if t.pos_=="VERB")
def negs(text): return Counter(t.text.lower() for t in nlp(str(text)) if t.dep_=="neg" or t.text.lower() in {"no","not","n't","never","neither","nor"})
def content_counter(text): return Counter(t.lemma_.lower() for t in nlp(str(text)) if t.is_alpha and not t.is_stop and t.pos_ in {"NOUN","PROPN","VERB","ADJ","ADV"})
def recall_counter(a,b):
    A0=content_counter(a); B0=content_counter(b); n=sum(A0.values())
    return sum(min(v,B0[k]) for k,v in A0.items())/n if n else 1.


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|█████████████████████████████████████████████████████████████████████████████| 148/148 [00:00<00:00, 11915.64it/s]


In [5]:
# Generate valid FUSE / ADD_COREF / ADD_RESTORE candidates. No endpoint selection yet.

fusion_candidates=[]
for i,r in enumerate(fusion_manifest.itertuples(index=False),1):
    src=r.sentence_a+" "+r.sentence_b
    prompt=f"""Fuse the following TWO adjacent sentences into exactly ONE grammatical sentence.
Preserve every fact, organization name, other named entity, number, verb meaning, negation, tense, and modality.
Do not add, delete, repeat, abbreviate, or rename any named entity. Keep every organization surface form exactly as written.
Sentence 1: {r.sentence_a}
Sentence 2: {r.sentence_b}"""
    for rank,cand in enumerate(generate(prompt),1):
        reason=""
        if cand=="NO_CHANGE": reason="no_change"
        elif len(list(nlp(cand).sents))!=1: reason="not_one_sentence"
        elif entity_counter(cand)!=entity_counter(src): reason="entity_multiset_changed"
        elif digits(cand)!=digits(src): reason="numbers_changed"
        elif negs(cand)!=negs(src): reason="negation_changed"
        elif any(verbs(cand)[v]<n for v,n in verbs(src).items()): reason="verb_lost"
        elif recall_counter(src,cand)<.92: reason="content_recall"
        fusion_candidates.append({**r._asdict(),"rank":rank,"candidate":cand,"valid":not reason,"reason":reason,
                                  "logp_delta":mean_logp(cand)-mean_logp(src) if not reason else np.nan})
    if i%10==0 or i==len(fusion_manifest): print("fusion",i,"/",len(fusion_manifest))
fc=pd.DataFrame(fusion_candidates)

coref_candidates=[]
for i,r in enumerate(coref_manifest.itertuples(index=False),1):
    prompt=f"""Use the first sentence only as context. Rewrite ONLY the second sentence.
If a pronoun or generic organization reference in Sentence 2 unambiguously refers to "{r.target_org}", replace exactly ONE such reference with the exact organization name "{r.target_org}".
Otherwise return NO_CHANGE. Preserve every fact, number, other named entity, verb meaning, negation, tense, and modality. Keep exactly one sentence.
Context: {r.context_sentence}
Sentence 2: {r.source_sentence}"""
    src=entity_counter(r.source_sentence); target=("ORG",norm_surface(r.target_org))
    for rank,cand in enumerate(generate(prompt),1):
        reason=""; ec=entity_counter(cand) if cand!="NO_CHANGE" else Counter()
        if cand=="NO_CHANGE": reason="no_change"
        elif len(list(nlp(cand).sents))!=1: reason="not_one_sentence"
        elif ec[target]!=src[target]+1: reason="target_ORG_not_plus_one"
        elif any(ec[k]<n for k,n in src.items()): reason="old_entity_lost"
        elif any(n-src[k]>0 and k!=target for k,n in ec.items()): reason="other_entity_added"
        elif digits(cand)!=digits(r.source_sentence): reason="numbers_changed"
        elif negs(cand)!=negs(r.source_sentence): reason="negation_changed"
        elif any(verbs(cand)[v]<n for v,n in verbs(r.source_sentence).items()): reason="verb_lost"
        elif recall_counter(r.source_sentence,cand)<.90: reason="content_recall"
        coref_candidates.append({**r._asdict(),"rank":rank,"candidate":cand,"valid":not reason,"reason":reason,
                                 "logp_delta":mean_logp(cand)-mean_logp(r.source_sentence) if not reason else np.nan})
    if i%10==0 or i==len(coref_manifest): print("coref",i,"/",len(coref_manifest))
cc=pd.DataFrame(coref_candidates)

restore_candidates=[]
for i,r in enumerate(restore_manifest.itertuples(index=False),1):
    prompt=f"""The original sentence contained the organization "{r.target_org}".
Rewrite the CURRENT sentence minimally so that the exact organization name "{r.target_org}" is restored ONLY if it expresses the same original proposition shown below.
Do not add any other fact, name, number, comparison, date, or measurement. Return exactly one sentence, or NO_CHANGE.
Original sentence: {r.original_sentence}
Current sentence: {r.source_sentence}"""
    src=entity_counter(r.source_sentence); target=("ORG",norm_surface(r.target_org))
    for rank,cand in enumerate(generate(prompt),1):
        reason=""; ec=entity_counter(cand) if cand!="NO_CHANGE" else Counter()
        if cand=="NO_CHANGE": reason="no_change"
        elif len(list(nlp(cand).sents))!=1: reason="not_one_sentence"
        elif ec[target]!=src[target]+1: reason="target_ORG_not_plus_one"
        elif any(ec[k]<n for k,n in src.items()): reason="old_entity_lost"
        elif any(n-src[k]>0 and k!=target for k,n in ec.items()): reason="other_entity_added"
        elif digits(cand)!=digits(r.source_sentence): reason="numbers_changed"
        elif negs(cand)!=negs(r.source_sentence): reason="negation_changed"
        elif any(verbs(cand)[v]<n for v,n in verbs(r.source_sentence).items()): reason="verb_lost"
        elif recall_counter(r.source_sentence,cand)<.90: reason="content_recall"
        restore_candidates.append({**r._asdict(),"rank":rank,"candidate":cand,"valid":not reason,"reason":reason,
                                   "logp_delta":mean_logp(cand)-mean_logp(r.source_sentence) if not reason else np.nan})
    if i%10==0 or i==len(restore_manifest): print("restore",i,"/",len(restore_manifest))
rc=pd.DataFrame(restore_candidates)

for name,x in [("FUSE",fc),("COREF",cc),("RESTORE",rc)]:
    print("\n",name,"valid:",int(x.valid.sum()) if len(x) else 0)
    if len(x): display(x.reason.value_counts(dropna=False).to_frame("n"))


fusion 10 / 28
fusion 20 / 28
fusion 28 / 28
coref 10 / 13
coref 13 / 13
restore 10 / 10

 FUSE valid: 51


,n
reason,
entity_multiset_changed,121
,51
not_one_sentence,42
no_change,8
verb_lost,1



 COREF valid: 6


,n
reason,
target_ORG_not_plus_one,69
no_change,13
old_entity_lost,12
,6
content_recall,2
other_entity_added,1
verb_lost,1



 RESTORE valid: 0


,n
reason,
target_ORG_not_plus_one,49
old_entity_lost,19
no_change,10
verb_lost,1
not_one_sentence,1


In [6]:
# New genre-routed REMOVE and SPLIT manifests/candidates.

remove_rows=[]; split_rows=[]
for r in data.itertuples(index=False):
    doc=nlp(r.text); sents=list(doc.sents)
    for i,s in enumerate(sents):
        if i>0:
            prior=" ".join(x.text for x in sents[max(0,i-2):i])
            for ent in [e for e in s.ents if e.label_=="ORG" and re.search(rf"(?i)(?<!\w){re.escape(e.text)}(?!\w)",prior)]:
                remove_rows.append({"case_id":r.case_id,"domain":r.domain,"sent_i":i,"start":s.start_char,"end":s.end_char,
                                    "context":" ".join(x.text for x in sents[max(0,i-2):i]),
                                    "source_sentence":s.text,"target_org":ent.text})
        orgs=[e.text for e in s.ents if e.label_=="ORG"]
        if len(orgs)>=2 and len(s.text.split())<=70:
            split_rows.append({"case_id":r.case_id,"domain":r.domain,"sent_i":i,"start":s.start_char,"end":s.end_char,
                               "source_sentence":s.text,"orgs":" | ".join(orgs)})

remove_manifest=pd.DataFrame(remove_rows); split_manifest=pd.DataFrame(split_rows)
print("REMOVE:",len(remove_manifest),"candidates /",remove_manifest.case_id.nunique() if len(remove_manifest) else 0,"cases")
print("SPLIT:",len(split_manifest),"candidates /",split_manifest.case_id.nunique() if len(split_manifest) else 0,"cases")

remove_candidates=[]
for i,r in enumerate(remove_manifest.itertuples(index=False),1):
    prompt=f"""Use the context only to resolve reference. Rewrite ONLY the target sentence.
The organization "{r.target_org}" has already been explicitly named in the preceding context.
Replace exactly ONE occurrence of "{r.target_org}" with a natural reduced reference such as a pronoun or generic organizational noun.
Preserve every fact, number, other named entity, verb meaning, negation, tense, and modality. Keep exactly one sentence. Add no new fact or entity.
Return only the rewritten sentence, or NO_CHANGE if no natural replacement exists.
Context: {r.context}
Target sentence: {r.source_sentence}"""
    src=entity_counter(r.source_sentence); target=("ORG",norm_surface(r.target_org))
    for rank,cand in enumerate(generate(prompt),1):
        reason=""; ec=entity_counter(cand) if cand!="NO_CHANGE" else Counter()
        if cand=="NO_CHANGE": reason="no_change"
        elif len(list(nlp(cand).sents))!=1: reason="sentence_count"
        elif ec[target]!=src[target]-1: reason="target_ORG_not_minus_one"
        elif any(ec[k]!=n for k,n in src.items() if k!=target): reason="other_entity_changed"
        elif any(k not in src and n>0 for k,n in ec.items()): reason="new_entity"
        elif digits(cand)!=digits(r.source_sentence): reason="numbers_changed"
        elif negs(cand)!=negs(r.source_sentence): reason="negation_changed"
        elif any(verbs(cand)[v]<n for v,n in verbs(r.source_sentence).items()): reason="verb_lost"
        elif recall_counter(r.source_sentence,cand)<.88: reason="content_recall"
        remove_candidates.append({**r._asdict(),"rank":rank,"candidate":cand,"valid":not reason,"reason":reason,
                                  "logp_delta":mean_logp(cand)-mean_logp(r.source_sentence) if not reason else np.nan})
    if i%10==0 or i==len(remove_manifest): print("remove",i,"/",len(remove_manifest))
dc=pd.DataFrame(remove_candidates)

split_candidates=[]
for i,r in enumerate(split_manifest.itertuples(index=False),1):
    prompt=f"""Split the sentence below into exactly TWO natural consecutive sentences.
Preserve every fact, organization name, other named entity, number, verb meaning, negation, tense, and modality.
Do not add, delete, repeat, abbreviate, or rename any named entity. Keep every organization surface form exactly as written.
Return only the two rewritten sentences.
Sentence: {r.source_sentence}"""
    src=entity_counter(r.source_sentence)
    for rank,cand in enumerate(generate(prompt),1):
        reason=""
        if cand=="NO_CHANGE": reason="no_change"
        elif len(list(nlp(cand).sents))!=2: reason="not_two_sentences"
        elif entity_counter(cand)!=src: reason="entity_multiset_changed"
        elif digits(cand)!=digits(r.source_sentence): reason="numbers_changed"
        elif negs(cand)!=negs(r.source_sentence): reason="negation_changed"
        elif any(verbs(cand)[v]<n for v,n in verbs(r.source_sentence).items()): reason="verb_lost"
        elif recall_counter(r.source_sentence,cand)<.92: reason="content_recall"
        split_candidates.append({**r._asdict(),"rank":rank,"candidate":cand,"valid":not reason,"reason":reason,
                                 "logp_delta":mean_logp(cand)-mean_logp(r.source_sentence) if not reason else np.nan})
    if i%10==0 or i==len(split_manifest): print("split",i,"/",len(split_manifest))
sc=pd.DataFrame(split_candidates)

for name,x in [("REMOVE",dc),("SPLIT",sc)]:
    print("\n",name,"valid:",int(x.valid.sum()) if len(x) else 0)
    if len(x): display(x.reason.value_counts(dropna=False).to_frame("n"))


REMOVE: 29 candidates / 14 cases
SPLIT: 35 candidates / 20 cases
remove 10 / 29
remove 20 / 29
remove 29 / 29
split 10 / 35
split 20 / 35
split 30 / 35
split 35 / 35

 REMOVE valid: 18


,n
reason,
target_ORG_not_minus_one,109
other_entity_changed,43
no_change,24
new_entity,18
,18
verb_lost,7
content_recall,4
sentence_count,3
negation_changed,3



 SPLIT valid: 67


,n
reason,
not_two_sentences,127
entity_multiset_changed,84
,67
negation_changed,1


In [7]:
# Unified genre-conditioned Step-10.9 controller.

def valid_pool(df,op):
    if df is None or not len(df): return []
    x=df.loc[df.valid].copy(); x["op"]=op
    return x.to_dict("records")

pool=(valid_pool(cc,"ADD_COREF")+valid_pool(rc,"ADD_RESTORE")+valid_pool(fc,"FUSE")+
      valid_pool(dc,"REMOVE")+valid_pool(sc,"SPLIT"))
OP_CLASS={"ADD_COREF":"ADD","ADD_RESTORE":"ADD","REMOVE":"REMOVE","FUSE":"FUSE","SPLIT":"SPLIT"}

def overlaps(a,b): return max(int(a["start"]),int(b["start"]))<min(int(a["end"]),int(b["end"]))
def render(base,selected):
    text=base
    for q in sorted(selected,key=lambda z:int(z["start"]),reverse=True):
        text=text[:int(q["start"])]+q["candidate"]+text[int(q["end"]):]
    return text
def no_good_feature_escape(before,after,domain):
    return all(not inside_band(before[f],domain,f) or inside_band(after[f],domain,f) for f in ORG_FEATURES)

genre_rows=[]; accepted_genre=[]
for r in data.itertuples(index=False):
    base=r.text; allowed=set(GENRE_POLICY[r.domain]["ops"])
    candidates=[q for q in pool if q["case_id"]==r.case_id and OP_CLASS[q["op"]] in allowed]
    selected=[]; current=base; current_stats=org_stats(current); current_d=org_band_distance(current,r.domain)
    max_edits=10 if r.domain in {"books","reviews"} else 5
    for _ in range(max_edits):
        best=None
        for q in candidates:
            if q in selected or any(overlaps(q,z) for z in selected): continue
            trial_sel=selected+[q]; trial=render(base,trial_sel); trial_stats=org_stats(trial)
            d=org_band_distance(trial,r.domain); gain=current_d-d
            if gain<=1e-12 or not no_good_feature_escape(current_stats,trial_stats,r.domain): continue
            lp=float(q.get("logp_delta",np.nan)); key=(gain,lp if np.isfinite(lp) else -999.)
            if best is None or key>best[0]: best=(key,q,trial,trial_stats,d)
        if best is None: break
        selected.append(best[1]); current,current_stats,current_d=best[2],best[3],best[4]

    genre_rows.append({"case_id":r.case_id,"source_id":r.source_id,"domain":r.domain,"text":current,
                       "edited":bool(selected),"n_edits":len(selected),"distance_before":org_band_distance(base,r.domain),
                       "distance_after":current_d,"ops":" | ".join(q["op"] for q in selected)})
    for j,q in enumerate(selected,1):
        accepted_genre.append({"case_id":r.case_id,"domain":r.domain,"edit_index":j,"op":q["op"],
                               "candidate":q["candidate"],"logp_delta":q.get("logp_delta",np.nan)})

genre_arm=pd.DataFrame(genre_rows); genre_edits=pd.DataFrame(accepted_genre)
print("Edited:",genre_arm.edited.sum(),"/ 54")
print("Total edits:",genre_arm.n_edits.sum())
display(genre_arm.groupby("domain").agg(cases=("case_id","size"),edited=("edited","sum"),edits=("n_edits","sum"),
                                         distance_before=("distance_before","mean"),distance_after=("distance_after","mean")))

rows=[]
for domain,g in genre_arm.groupby("domain"):
    before=data.loc[data.domain.eq(domain)]
    sb=pd.DataFrame([org_stats(t) for t in before.text]); sa=pd.DataFrame([org_stats(t) for t in g.text])
    for f in ORG_FEATURES:
        rows.append({"domain":domain,"feature":f,"AI_before_median":sb[f].median(),"AI_after_median":sa[f].median(),
                     "Human_median":hmed.loc[domain,f],"before_gap":sb[f].median()-hmed.loc[domain,f],
                     "after_gap":sa[f].median()-hmed.loc[domain,f]})
genre_effect=pd.DataFrame(rows)
display(genre_effect)
if len(genre_edits): display(genre_edits)


Edited: 8 / 54
Total edits: 9


,cases,edited,edits,distance_before,distance_after
domain,,,,,
abstracts,10,0,0,0.237388,0.237388
books,7,0,0,0.454840,0.454840
news,12,0,0,0.974793,0.974793
recipes,9,2,2,2.969949,2.912995
reddit,6,0,0,0.149863,0.149863
reviews,1,0,0,0.000000,0.000000
wiki,9,6,7,1.085581,0.957903


,domain,feature,AI_before_median,AI_after_median,Human_median,before_gap,after_gap
0,abstracts,ner_ORG_entity_share,0.361111,0.361111,0.400000,-0.038889,-0.038889
1,abstracts,ner_ORG_sentence_mean,0.366667,0.366667,0.375000,-0.008333,-0.008333
2,abstracts,ner_ORG_sentence_std,0.480651,0.480651,0.489898,-0.009247,-0.009247
3,abstracts,ner_ORG_sentence_max,1.000000,1.000000,1.000000,0.000000,0.000000
4,books,ner_ORG_entity_share,0.000000,0.000000,0.175192,-0.175192,-0.175192
5,books,ner_ORG_sentence_mean,0.000000,0.000000,0.436508,-0.436508,-0.436508
6,books,ner_ORG_sentence_std,0.000000,0.000000,0.476565,-0.476565,-0.476565
7,books,ner_ORG_sentence_max,0.000000,0.000000,1.000000,-1.000000,-1.000000
8,news,ner_ORG_entity_share,0.150997,0.150997,0.176471,-0.025473,-0.025473
9,news,ner_ORG_sentence_mean,0.266667,0.266667,0.285714,-0.019048,-0.019048


,case_id,domain,edit_index,op,candidate,logp_delta
0,gate_0018,recipes,1,SPLIT,I had the World's Best Key Lime Pie. It was pr...,0.654278
1,gate_0021,wiki,1,SPLIT,"In addition to his academic work, Gruzman has ...",0.345020
2,gate_0027,wiki,1,SPLIT,The Scottish Mining Museum in Newtongrange att...,0.051933
3,gate_0051,wiki,1,SPLIT,In a two thousand fourteen interview with The ...,-0.161999
4,gate_0078,wiki,1,REMOVE,"Throughout its history, it has been primarily ...",0.519115
5,gate_0078,wiki,2,REMOVE,The community had a population of around 200 r...,0.057602
6,gate_0082,wiki,1,SPLIT,He has won numerous awards for his composition...,0.463235
7,gate_0085,recipes,1,SPLIT,Step 9 Serve immediately. Enjoy your yummy Bro...,-0.125779
8,gate_0086,wiki,1,SPLIT,He joined the Labor Party during his youth. He...,0.082960


In [8]:
# Save exploratory Step-10.9 outputs. Do not call this accepted until frozen-FZ301 evaluation.

contract={"name":"C04d_ORG_genre_conditioned_step10p9","seed":SEED,"input_step10":str(STEP10_RUN.relative_to(ROOT)),
          "qwen":QWEN_ID,"qwen_revision":QWEN_REV,"policy":"same-genre human Q20-Q80 ORG bands",
          "operations":["ADD_COREF","ADD_RESTORE","FUSE","REMOVE","SPLIT"]}
run=hashlib.sha256(json.dumps(contract,sort_keys=True).encode()).hexdigest()[:12]
OUT=OUT_ROOT/run; OUT.mkdir(parents=True,exist_ok=True)
(OUT/"contract.json").write_text(json.dumps(contract,indent=2))
for name,obj in {
    "genre_medians":genre_medians,"genre_policy":policy_df,
    "fusion_manifest":fusion_manifest,"fusion_candidates":fc,
    "coref_manifest":coref_manifest,"coref_candidates":cc,
    "restore_manifest":restore_manifest,"restore_candidates":rc,
    "remove_manifest":remove_manifest,"remove_candidates":dc,
    "split_manifest":split_manifest,"split_candidates":sc,
    "alias_audit":alias_audit,"texts_after_genre_ORG_step10p9":genre_arm,
    "accepted_edits":genre_edits,"genre_effect":genre_effect
}.items():
    obj.to_csv(OUT/f"{name}.csv",index=False)
print("Saved:",OUT.relative_to(ROOT))
print("Next: inspect policy + genre_effect; only then run frozen-FZ301 TPR/contribution evaluation.")


Saved: results/causal/C04d_ORG_discourse_step10p9/42d8c051258b
Next: inspect policy + genre_effect; only then run frozen-FZ301 TPR/contribution evaluation.


In [9]:
# Diagnose Step-10.9 candidate starvation by genre, especially books/reviews.

ops={"ADD_COREF":cc,"ADD_RESTORE":rc,"FUSE":fc,"REMOVE":dc,"SPLIT":sc}
rows=[]
for op,df in ops.items():
    if df is None or not len(df): continue
    for domain,g in df.groupby("domain"):
        rows.append({
            "operation":op,"domain":domain,
            "generated":len(g),
            "valid":int(g.valid.sum()),
            "cases_with_valid":g.loc[g.valid,"case_id"].nunique()
        })
coverage=pd.DataFrame(rows)
display(coverage.sort_values(["domain","operation"]).reset_index(drop=True))

# Compare current Step-10 ORGs with ORGs still available in the original source.
rows=[]
for r in data.itertuples(index=False):
    cur_doc=nlp(r.text)
    src=text_map[r.source_id]
    src_doc=nlp(src)

    cur_orgs=sorted({e.text for e in cur_doc.ents if e.label_=="ORG"})
    src_orgs=sorted({e.text for e in src_doc.ents if e.label_=="ORG"})
    missing=[x for x in src_orgs if x.lower() not in {y.lower() for y in cur_orgs}]

    rows.append({
        "case_id":r.case_id,"domain":r.domain,
        "current_n_ORG":len(cur_orgs),
        "source_n_ORG":len(src_orgs),
        "missing_source_n_ORG":len(missing),
        "current_ORGs":" | ".join(cur_orgs),
        "missing_source_ORGs":" | ".join(missing)
    })

source_org_audit=pd.DataFrame(rows)

print("Books / reviews:")
display(
    source_org_audit.loc[
        source_org_audit.domain.isin(["books","reviews"])
    ].reset_index(drop=True)
)

print("\nAll genres — source ORG restoration capacity:")
display(
    source_org_audit.groupby("domain").agg(
        cases=("case_id","size"),
        zero_current_ORG=("current_n_ORG",lambda x:(x==0).sum()),
        cases_with_missing_source_ORG=("missing_source_n_ORG",lambda x:(x>0).sum()),
        missing_source_ORGs=("missing_source_n_ORG","sum")
    )
)

,operation,domain,generated,valid,cases_with_valid
0,ADD_COREF,abstracts,8,0,0
1,ADD_RESTORE,abstracts,8,0,0
2,FUSE,abstracts,56,16,2
3,REMOVE,abstracts,56,7,4
4,SPLIT,abstracts,40,10,2
5,ADD_COREF,books,8,0,0
6,FUSE,books,16,4,1
7,SPLIT,books,8,1,1
8,ADD_COREF,news,48,0,0
9,ADD_RESTORE,news,16,0,0


Books / reviews:


,case_id,domain,current_n_ORG,source_n_ORG,missing_source_n_ORG,current_ORGs,missing_source_ORGs
0,gate_0016,books,0,0,0,,
1,gate_0020,reviews,0,0,0,,
2,gate_0023,books,0,0,0,,
3,gate_0035,books,0,0,0,,
4,gate_0047,books,4,2,0,Eomer | Plot Summary | Wall | the First King's...,
5,gate_0065,books,0,0,0,,
6,gate_0070,books,0,0,0,,
7,gate_0075,books,1,1,0,Iyanthira,



All genres — source ORG restoration capacity:


,cases,zero_current_ORG,cases_with_missing_source_ORG,missing_source_ORGs
domain,,,,
abstracts,10,2,5,8
books,7,5,0,0
news,12,3,7,10
recipes,9,2,5,6
reddit,6,4,2,3
reviews,1,1,0,0
wiki,9,1,4,6


In [10]:
# Why did source-grounded ORG restoration fail in abstracts/news?

for domain in ["abstracts","news"]:
    x=rc.loc[rc.domain.eq(domain)].copy()
    print(f"\n{domain.upper()} — {len(x)} generated candidates")
    display(x.reason.value_counts(dropna=False).to_frame("n"))

    cols=[
        "case_id","target_org","original_sentence","source_sentence",
        "candidate","reason","logp_delta"
    ]
    display(x[cols].head(50))


ABSTRACTS — 8 generated candidates


,n
reason,
target_ORG_not_plus_one,5
no_change,1
verb_lost,1
old_entity_lost,1


,case_id,target_org,original_sentence,source_sentence,candidate,reason,logp_delta
48,gate_0069,Multiple-Input Multiple-Output,This paper investigates resource allocation in...,This paper explores resource allocation in MIM...,NO_CHANGE,no_change,NaN
49,gate_0069,Multiple-Input Multiple-Output,This paper investigates resource allocation in...,This paper explores resource allocation in MIM...,The original sentence contained the organizati...,target_ORG_not_plus_one,NaN
50,gate_0069,Multiple-Input Multiple-Output,This paper investigates resource allocation in...,This paper explores resource allocation in MIM...,Multiple-Input Multiple-Output (MIMO),verb_lost,NaN
51,gate_0069,Multiple-Input Multiple-Output,This paper investigates resource allocation in...,This paper explores resource allocation in MIM...,Multiple-Input Multiple-Output,old_entity_lost,NaN
52,gate_0069,Multiple-Input Multiple-Output,This paper investigates resource allocation in...,This paper explores resource allocation in MIM...,NO_CHANGE.,target_ORG_not_plus_one,NaN
53,gate_0069,Multiple-Input Multiple-Output,This paper investigates resource allocation in...,This paper explores resource allocation in MIM...,No change.,target_ORG_not_plus_one,NaN
54,gate_0069,Multiple-Input Multiple-Output,This paper investigates resource allocation in...,This paper explores resource allocation in MIM...,No change needed.,target_ORG_not_plus_one,NaN
55,gate_0069,Multiple-Input Multiple-Output,This paper investigates resource allocation in...,This paper explores resource allocation in MIM...,No change required.,target_ORG_not_plus_one,NaN



NEWS — 16 generated candidates


,n
reason,
target_ORG_not_plus_one,14
no_change,2


,case_id,target_org,original_sentence,source_sentence,candidate,reason,logp_delta
0,gate_0003,Transparency International,The initiative is part of a broader anti-corru...,The initiative is part of a broader anti-corru...,NO_CHANGE,no_change,NaN
1,gate_0003,Transparency International,The initiative is part of a broader anti-corru...,The initiative is part of a broader anti-corru...,The original sentence contained the organizati...,target_ORG_not_plus_one,NaN
2,gate_0003,Transparency International,The initiative is part of a broader anti-corru...,The initiative is part of a broader anti-corru...,"The organization ""Transparency International"" ...",target_ORG_not_plus_one,NaN
3,gate_0003,Transparency International,The initiative is part of a broader anti-corru...,The initiative is part of a broader anti-corru...,"The organization ""Transparency International"" ...",target_ORG_not_plus_one,NaN
4,gate_0003,Transparency International,The initiative is part of a broader anti-corru...,The initiative is part of a broader anti-corru...,The original sentence contained the organizati...,target_ORG_not_plus_one,NaN
5,gate_0003,Transparency International,The initiative is part of a broader anti-corru...,The initiative is part of a broader anti-corru...,No change.,target_ORG_not_plus_one,NaN
6,gate_0003,Transparency International,The initiative is part of a broader anti-corru...,The initiative is part of a broader anti-corru...,NO_CHANGE.,target_ORG_not_plus_one,NaN
7,gate_0003,Transparency International,The initiative is part of a broader anti-corru...,The initiative is part of a broader anti-corru...,No change needed.,target_ORG_not_plus_one,NaN
32,gate_0054,Academy Award,The yet untitled project will be directed by D...,The yet untitled project will be directed by D...,NO_CHANGE,no_change,NaN
33,gate_0054,Academy Award,The yet untitled project will be directed by D...,The yet untitled project will be directed by D...,The yet untitled project will be directed by D...,target_ORG_not_plus_one,NaN


In [11]:
# Rebuild ADD_RESTORE coverage with broader but still source-grounded alignment.

from difflib import SequenceMatcher

ADD_DOMAINS={g for g,p in GENRE_POLICY.items() if "ADD" in p["ops"]}
restore_rows=[]

def initials(s):
    words=[w for w in re.findall(r"[A-Za-z]+",s) if w.lower() not in {"the","of","and","for","in","to","a","an"}]
    return "".join(w[0].upper() for w in words) if len(words)>=2 else ""

for r in data.itertuples(index=False):
    if r.domain not in ADD_DOMAINS: continue
    cur_doc=nlp(r.text); src_doc=nlp(text_map[r.source_id])
    cur_orgs={norm_surface(e.text) for e in cur_doc.ents if e.label_=="ORG"}

    for ss in src_doc.sents:
        for ent in [e for e in ss.ents if e.label_=="ORG" and norm_surface(e.text) not in cur_orgs]:
            A0=content_lemmas(ss.text)-content_lemmas(ent.text)
            ac=initials(ent.text)
            candidates=[]

            for j,cs in enumerate(cur_doc.sents):
                if norm_surface(ent.text) in norm_surface(cs.text): continue
                B0=content_lemmas(cs.text)
                overlap=len(A0&B0)
                jac=overlap/max(1,len(A0|B0))
                seq=SequenceMatcher(None,norm_surface(ss.text),norm_surface(cs.text)).ratio()
                verbA={t.lemma_.lower() for t in nlp(ss.text) if t.pos_=="VERB"}
                verbB={t.lemma_.lower() for t in nlp(cs.text) if t.pos_=="VERB"}
                verb_overlap=len(verbA&verbB)
                alias=bool(ac and re.search(rf"\b{re.escape(ac)}\b",cs.text))
                score=.55*jac+.30*seq+.15*min(verb_overlap,1)+(.20 if alias else 0)

                if alias or (overlap>=3 and (jac>=.25 or seq>=.55)):
                    candidates.append((score,j,cs,jac,seq,alias,ac))

            for score,j,cs,jac,seq,alias,ac in sorted(candidates,reverse=True)[:2]:
                restore_rows.append({
                    "case_id":r.case_id,"domain":r.domain,"sent_i":j,
                    "start":cs.start_char,"end":cs.end_char,
                    "target_org":ent.text,"original_sentence":ss.text,
                    "source_sentence":cs.text,"align_score":score,
                    "jaccard":jac,"sequence_ratio":seq,
                    "alias_present":alias,"alias":ac
                })

restore_manifest=pd.DataFrame(restore_rows).drop_duplicates(
    ["case_id","sent_i","target_org"]
).reset_index(drop=True)

print("New restoration coverage:")
display(
    restore_manifest.groupby("domain").agg(
        candidates=("case_id","size"),
        cases=("case_id","nunique"),
        target_ORGs=("target_org","nunique")
    )
)
display(restore_manifest)

New restoration coverage:


,candidates,cases,target_ORGs
domain,,,
abstracts,6,4,6
news,3,3,3


,case_id,domain,sent_i,start,end,target_org,original_sentence,source_sentence,align_score,jaccard,sequence_ratio,alias_present,alias
0,gate_0003,news,6,829,917,Transparency International,The initiative is part of a broader anti-corru...,The initiative is part of a broader anti-corru...,0.776274,0.777778,0.661654,False,TI
1,gate_0028,abstracts,3,473,628,3D ResNets,"Specifically, instead of operating only in the...",The study combines a convolutional autoencoder...,0.480427,0.323529,0.508287,False,DR
2,gate_0054,news,4,509,724,Academy Award,The yet untitled project will be directed by D...,The yet untitled project will be directed by D...,0.585366,0.600000,0.351220,False,AA
3,gate_0066,news,2,239,340,Parmalat,Parmalat's chief executive says a 23m-euro dea...,Tanzi's empire is being sold to Brazilian riva...,0.461634,0.318182,0.455446,False,
4,gate_0069,abstracts,0,0,111,Multiple-Input Multiple-Output,This paper investigates resource allocation in...,This paper explores resource allocation in MIM...,0.787159,0.642857,0.778626,True,MIMO
5,gate_0079,abstracts,0,0,89,Searching Multi-level,"The abstract for the academic paper titled ""UX...",Medical image segmentation is crucial for clin...,0.531117,0.421053,0.498462,False,SML
6,gate_0079,abstracts,0,0,89,Feature Aggregation,"The abstract for the academic paper titled ""UX...",Medical image segmentation is crucial for clin...,0.519538,0.400000,0.498462,False,FA
7,gate_0079,abstracts,4,494,870,SYNAPSE-BREAST-CT,Extensive experiments on three benchmark datas...,This mechanism enhances the spatial informatio...,0.398299,0.270270,0.332168,False,SBC
8,gate_0090,abstracts,6,1004,1084,https://github.com/VITA-Group/UA-SE,Our code and dataset are available at https://...,The code and two datasets are available at htt...,0.478977,0.375000,0.909091,False,HGCVGUS


In [12]:
# # Source-grounded restoration with deterministic alias expansion first,
# # then beam search with the exact ORG phrase lexically forced.

# def generate_forced(prompt,phrase,n=8,max_new=160):
#     msgs=[
#         {"role":"system","content":"You are a precise sentence editor. Return only the rewritten sentence."},
#         {"role":"user","content":prompt}
#     ]
#     txt=qtok.apply_chat_template(msgs,tokenize=False,add_generation_prompt=True)
#     enc=qtok(txt,return_tensors="pt",add_special_tokens=False).to(device)
#     L=enc.input_ids.shape[1]
#     forced=qtok(phrase,add_special_tokens=False).input_ids

#     with torch.inference_mode():
#         out=qwen.generate(
#             **enc,num_beams=n,num_return_sequences=n,max_new_tokens=max_new,
#             do_sample=False,early_stopping=True,force_words_ids=[forced],
#             pad_token_id=qtok.pad_token_id,eos_token_id=qtok.eos_token_id
#         )

#     vals=[x.strip().strip('"') for x in qtok.batch_decode(out[:,L:],skip_special_tokens=True)]
#     return list(dict.fromkeys(vals))

# def restoration_guard(src,cand,target):
#     old=entity_counter(src); new=entity_counter(cand); key=("ORG",norm_surface(target))
#     if len(list(nlp(cand).sents))!=1: return "not_one_sentence"
#     if new[key]!=old[key]+1: return "target_ORG_not_plus_one"
#     if any(new[k]<n for k,n in old.items()): return "old_entity_lost"
#     if any(n-old[k]>0 and k!=key for k,n in new.items()): return "other_entity_added"
#     if digits(cand)!=digits(src): return "numbers_changed"
#     if negs(cand)!=negs(src): return "negation_changed"
#     if any(verbs(cand)[v]<n for v,n in verbs(src).items()): return "verb_lost"
#     if recall_counter(src,cand)<.88: return "content_recall"
#     return ""

# restore_candidates=[]

# for i,r in enumerate(restore_manifest.itertuples(index=False),1):
#     proposals=[]

#     # Deterministic full-name expansion when an acronym already occurs.
#     if r.alias_present:
#         expanded=re.sub(
#             rf"\b{re.escape(r.alias)}\b",
#             f"{r.target_org} ({r.alias})",
#             r.source_sentence,count=1
#         )
#         proposals.append(("ALIAS_EXPAND",expanded))

#     prompt=f"""Rewrite the CURRENT sentence minimally so the exact organization name "{r.target_org}" appears exactly once.

# The organization is explicitly present in the original source sentence, so do not invent any fact.
# Preserve every current fact, number, other named entity, verb meaning, negation, tense, and modality.
# Keep exactly one sentence. Do not explain the edit.

# Original source sentence:
# {r.original_sentence}

# Current sentence:
# {r.source_sentence}"""

#     proposals += [("FORCED_QWEN",x) for x in generate_forced(prompt,r.target_org)]

#     for rank,(method,cand) in enumerate(proposals,1):
#         reason=restoration_guard(r.source_sentence,cand,r.target_org)
#         restore_candidates.append({
#             **r._asdict(),"rank":rank,"method":method,"candidate":cand,
#             "valid":not reason,"reason":reason,
#             "logp_delta":mean_logp(cand)-mean_logp(r.source_sentence) if not reason else np.nan
#         })

#     if i%10==0 or i==len(restore_manifest): print("restore",i,"/",len(restore_manifest))

# rc=pd.DataFrame(restore_candidates)

# print("\nRestoration results:")
# display(rc.reason.value_counts(dropna=False).to_frame("n"))

# print("\nValid coverage by genre:")
# display(
#     rc.loc[rc.valid].groupby("domain").agg(
#         valid_candidates=("case_id","size"),
#         cases=("case_id","nunique"),
#         target_ORGs=("target_org","nunique")
#     )
# )

# display(
#     rc.loc[rc.valid,[
#         "case_id","domain","target_org","method",
#         "original_sentence","source_sentence","candidate","logp_delta"
#     ]]
# )

[transformers] Constrained Beam Search was moved to a `custom_generate` repo: https://hf.co/transformers-community/constrained-beam-search. To prevent loss of backward compatibility, add `custom_generate='transformers-community/constrained-beam-search'` to your `generate` call before v4.62.0.


ValueError: Constrained Beam Search requires `trust_remote_code=True` in your `generate` call, since it loads https://hf.co/transformers-community/constrained-beam-search.

In [13]:
# Inspect the 5 book cases with zero detected ORG after Step 10.

book_rows=[]
for r in data.loc[data.domain.eq("books")].itertuples(index=False):
    cur_doc=nlp(r.text)
    src=text_map[r.source_id]
    src_doc=nlp(src)

    cur_orgs=[e.text for e in cur_doc.ents if e.label_=="ORG"]
    src_orgs=[e.text for e in src_doc.ents if e.label_=="ORG"]

    if len(cur_orgs)==0:
        book_rows.append({
            "case_id":r.case_id,
            "source_id":r.source_id,
            "source_ORGs":" | ".join(src_orgs) or "(none)",
            "current_ORGs":" | ".join(cur_orgs) or "(none)",
            "SOURCE_TEXT":src,
            "CURRENT_STEP10_TEXT":r.text,
        })

books_zero=pd.DataFrame(book_rows).head(5)

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", None)
display(books_zero)

,case_id,source_id,source_ORGs,current_ORGs,SOURCE_TEXT,CURRENT_STEP10_TEXT
0,gate_0016,ai__raid:296752cd-3b82-479f-b5a9-0bb1001ba706,(none),(none),"The novel follows the story of Stevens, a butler at Darlington Hall, who has dedicated his life to serving his master, Lord Darlington. The story takes place in the 1930s, and as the novel begins, Stevens is in his mid-50s and has been working at Darlington Hall for over 30 years. He is deeply devoted to his work and takes great pride in his profession, believing that a butler's role is not only to serve his master but also to provide a sense of continuity and tradition in a rapidly changing world.\n\nDespite his dedication, Stevens's life is not without its challenges. He has few close relationships outside of work, and his only real companion is his father, who is also a butler at a nearby estate. As the novel progresses, Stevens's father becomes ill, and Stevens is forced to confront the reality of his own mortality and the choices he has made in his life.\n\nAt the same time, Stevens becomes embroiled in a series of events that challenge his beliefs about his role as a butler and his relationship with his master. Lord Darlington is a wealthy and influential man, but he is also a Nazi sympathizer, and Stevens is forced to confront the moral implications of serving such a man. As the political situation in Europe becomes increasingly tense, Stevens must decide where his loyalties truly lie and what he is willing to do to uphold his sense of duty and honor.\n\nThroughout the novel, Stevens grapples with questions of identity, loyalty, and morality, all set against the backdrop of the rapidly changing world of the 1930s. As he reflects on his life and the choices he has made, he must confront the remains of his own day and the legacy he will leave behind. The novel is a poignant and thought-provoking exploration of the human condition, and it raises important questions about the nature of duty, loyalty, and identity.","The novel chronicles Stevens, a butler at Darlington Hall who has dedicated his life to serving his master Lord Darlington. Set in the 1930s, the story begins with Stevens in his mid-50s and has been working at Darlington Hall for over 30 years. He is deeply devoted to his work and takes great pride in his profession, believing that a butler's role extends beyond mere service to provide continuity and tradition in an evolving world. Despite his dedication, Stevens faces numerous challenges. He has few close relationships outside of work and his only companion is his father, who is also a butler at a nearby estate. As the novel progresses, Stevens's father becomes ill, forcing him to face the reality of his own mortality and the choices he has made in life. Simultaneously, Stevens finds himself entangled in events that challenge his beliefs concerning his role as a butler and his relationship with his two masters. Lord Darlington, a wealthy and influential man, is a Nazi sympathizer, prompting Stevens to grapple with moral implications of serving such a man. As political tensions escalate in Europe, Stevens must decide where his loyalties truly lie and what he is willing to do to uphold his sense of duty and honor. All over the novel, Stevens grapples with three issues: identity, loyalty, and morality, all against the backdrop of rapidly changing times in the 1930s. Reflecting on his life and past decisions, he confronts both personal and historical legacies. The narrative explores profound themes of human nature, duty, loyalty and identity, offering a poignant and thought-provoking examination of the human condition."
1,gate_0023,ai__raid:7b6ca4a8-9f9f-4ede-a9ce-4d6ca52ac525,(none),(none),"In the heart of an ancient, forgotten land lies a city that has been lost to time and legend. For centuries, adventurers have sought its mythical streets, searching for treasures beyond imagination and answers to mysteries long past. But none have ever found so much as a crumbling

In [14]:
# Show every named entity spaCy DOES detect in those same 5 book texts.

rows=[]
for r in books_zero.itertuples(index=False):
    for version,text in [("source",r.SOURCE_TEXT),("step10",r.CURRENT_STEP10_TEXT)]:
        ents=[f"{e.text} [{e.label_}]" for e in nlp(text).ents]
        rows.append({
            "case_id":r.case_id,
            "version":version,
            "entities":" | ".join(ents) or "(none)"
        })

display(pd.DataFrame(rows))

,case_id,version,entities
0,gate_0016,source,Stevens [PERSON] | Darlington Hall [FAC] | the 1930s [DATE] | Stevens [PERSON] | his mid-50s [DATE] | Darlington Hall [FAC] | 30 years [DATE] | Stevens [PERSON] | Stevens [PERSON] | Stevens [PERSON] | Stevens [PERSON] | Darlington [PERSON] | Nazi [NORP] | Stevens [PERSON] | Europe [LOC] | Stevens [PERSON] | Stevens [PERSON] | the 1930s [DATE]
1,gate_0016,step10,Stevens [PERSON] | Darlington Hall [FAC] | the 1930s [DATE] | Stevens [PERSON] | mid-50s [DATE] | Darlington Hall [FAC] | 30 years [DATE] | Stevens [PERSON] | Stevens [PERSON] | Stevens [PERSON] | two [CARDINAL] | Nazi [NORP] | Stevens [PERSON] | Europe [LOC] | Stevens [PERSON] | Stevens [PERSON] | three [CARDINAL] | the 1930s [DATE]
2,gate_0023,source,centuries [DATE] | the years [DATE] | Sarah [PERSON] | Sarah [PERSON] | Sarah [PERSON] | Sarah [PERSON] | Sarah [PERSON] | Sarah [PERSON] | one [CARDINAL]
3,gate_0023,step10,centuries [DATE] | two [CARDINAL] | the years [DATE] | Sarah [PERSON] | Sarah [PERSON] | Sarah [PERSON] | Sarah [PERSON] | Sarah [PERSON] | Sarah [PERSON] | two [CARDINAL] | one [CARDINAL]
4,gate_0035,source,The Last Children of Schewenborn [WORK_OF_ART] | German [NORP] | Schewenborn [GPE] | Peter [PERSON] | Peter [PERSON] | Peter [PERSON] | Peter [PERSON] | one [CARDINAL] | Lisa [PERSON] | Schewenborn [NORP] | Peter [PERSON] | Lisa [PERSON]
5,gate_0035,step10,The Last Children of Schewenborn [WORK_OF_ART] | German [NORP] | Schewenborn [GPE] | Peter [PERSON] | Peter [PERSON] | Peter [PERSON] | Peter [PERSON] | two [CARDINAL] | Lisa [PERSON] | Schewenborn [NORP] | Peter [PERSON] | Lisa [PERSON] | two [CARDINAL]
6,gate_0065,source,Detective Caul [WORK_OF_ART] | Caul [PERSON] | Detective Caul [PERSON] | Caul [PERSON] | Caul [PERSON] | Caul [PERSON] | Caul [PERSON] | Detective Caul [PERSON] | Detective Caul [PERSON] | Caul [PERSON] | two [CARDINAL] | one [CARDINAL] | Detective Caul [PERSON] | $3 billion and [MONEY] | $5 billion [MONEY] | Rowling [GPE] | Caul [WORK_OF_ART] | Detective [GPE] | Caul [PERSON] | Detective Caul [PERSON]
7,gate_0065,step10,two [CARDINAL] | Detective Caul [WORK_OF_ART] | Caul [PERSON] | Caul [PERSON] | Caul [PERSON] | Caul [PERSON] | Caul [PERSON] | Caul [PERSON] | Caul [PERSON] | Caul [PERSON] | Caul [PERSON] | two [CARDINAL] | Caul [PERSON] | Caul [PERSON] | $3 billion and [MONEY] | $5 billion [MONEY] | Rowling [GPE] | Caul [PERSON] | two [CARDINAL]
8,gate_0070,source,Dick Sand [PERSON] | first [ORDINAL] | Dick [PERSON] | Dick encounter's [PERSON] | Dick [PERSON] | Dick [PERSON] | Dick [PERSON]
9,gate_0070,step10,Dick Sand [PERSON] | first [ORDINAL] | Dick [PERSON] | two [CARDINAL] | Dick [PERSON] | Dick [PERSON] | two [CARDINAL] | Dick [PERSON] | Dick [PERSON]


In [15]:
# Inspect human "books" examples, especially those contributing ORGs.

rows=[]
for r in h.loc[h.domain.eq("books")].itertuples(index=False):
    doc=nlp(r.text)
    orgs=[e.text for e in doc.ents if e.label_=="ORG"]
    rows.append({
        "source_id":r.source_id,
        "n_ORG":len(orgs),
        "ORGs":" | ".join(orgs) or "(none)",
        "text":r.text
    })

human_books=pd.DataFrame(rows).sort_values("n_ORG",ascending=False).reset_index(drop=True)

print("Human books ORG-count distribution:")
display(human_books.n_ORG.value_counts().sort_index().to_frame("documents"))

print("\nFive human books with the most ORGs:")
pd.set_option("display.max_colwidth",None)
display(human_books.head(5))

Human books ORG-count distribution:


,documents
n_ORG,
0,4
1,2
2,2
3,1
4,1
5,2
6,1
7,2
8,3



Five human books with the most ORGs:


,source_id,n_ORG,ORGs,text
0,human__raid:0216216b-fd4f-403b-87f3-b1c52f29a3f1,8,"the Summer Country, Princess Guenevere | Morgause | Morgan | King Uther | Morgause | the King and Queen | Avalon | Guenevere","Raised in the tranquil beauty of the Summer Country, Princess Guenevere has had a charmed and contented life until the sudden, violent death of her mother, Queen Maire, leaves the Summer Country teetering on the brink of anarchy. Only the miraculous arrival of Arthur, heir to the Pendragon dynasty, allows Guenevere to claim her mother's throne. Smitten by the bold, sensuous princess, Arthur offers to marry her and unite their territories, while still allowing her to rule in her own right. Their love match creates the largest and most powerful kingdom in the isles. Arthur's glorious rule begins to crumble, however, when he is reunited with his mother and his long-lost half-sisters, Morgause and Morgan. Before Arthur's birth, his father - the savage and unscrupulous King Uther - banished his wife's young daughters, selling Morgause into a cruel marriage and imprisoning Morgan in a far-off convent. Both daughters have reason to avenge their suffering, but only one will strike the deadliest blows against the King and Queen, using her evil enchantments to destroy all Guenevere holds dear. When the Queen flees to Avalon, even her marriage with Arthur comes under threat. In the chaos that follows, a new young knight comes to Arthur's court to offer his services to the Queen. Her loyalty to Arthur betrayed, Guenevere falls in love with Lancelot, a love that may spell ruin for Camelot."
1,human__raid:0a9dc4af-7338-49c0-b3d9-3a1a3cf1606f,8,fed | Master Computer | Nila | Population Control Board | Nila | Robotic | Nila | Nila,"The plot opens in the year 2021.India is ruled by a dictator Jeeva. In his rule, population is kept under control, old people are allowed to live only up to particular age and are killed when they cross the limit, each and every one should have names with two letters only alloted by Government, every Citizen of the country must strictly adhere to the rule of Jeeva and admire,adore him, everywhere and anywhere lies Robots, people are identified only with their social security numbers and all the details are fed and controlled by Master Computer at Capital City. On a New Year eve, Nila, a homemaker lady is very delighted for having Government permission letter to have a boy baby from Population Control Board. She intimates the news to her husband Sibi. While Government, allots a home space of Nila's residence for a man Ravi and his pet Jeano. Jeano is a Robotic pet which could speak. As per government rule, home space should not be alloted for more than three people. So Nila's husband leaves to verify about Ravi's allotment but not returns back to home. Nila, to find him, uses his social security number but wonders to get an answer that there is no existence of such a person. She tries to find him but in vain. What happened to Sibi? What is the role of Ravi in Sibi's disappearance? How Nila finds her husband and a shocking truth about Jeeva and his dictatorship with the help of Jeano? The rest of story moves with the answers to these questions."
2,human__raid:1de9f910-c6a9-4bf8-8a2d-1f9c07390417,8,Bombaata | Amon-Rama | Akiro | Amon-Rama's | a Man-Ape | Bombaata | Taramis | Bombaata,"Princess Taramis of Shadizar promises to bring Conan's lost love Valeria back to life if the Cimmerian will procure two magical items that she hopes will gain her ultimate power, a wizard's gem and a horn that can awaken the dreaming god Dagoth. He undertakes the quest together with his thief partner Malak and Taramis' niece Jehnna and henchman Bombaata. On their journey they are joined by two additional allies whom Conan saves from dire fates; the magician Akiro and the female warrior Zula. At their goal, the castle of the wizard Amon-Rama, Jehnna is kidnapped. Thanks to Akiro's magic she is located in Amon-Rama's lair and a way in 